# **RAG BASED CHATBOT PROJECT**




# 1. Install everything

In [15]:
!pip install -q langchain langchain-community langchain-huggingface \
langchain-chroma langchain-groq chromadb pypdf gradio

# 2. Upload the PDFs

In [ ]:

https://event-management-five-lac.vercel.app/

# 3. Imports

In [16]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# 4. Load all PDFs

In [17]:
files = [
    "FAQs.pdf",
    "how-to-guides.pdf",
    "product-information.pdf"
]

docs = []

for file in files:
    docs.extend(PyPDFLoader(file).load())

print("Pages loaded:", len(docs))

Pages loaded: 29


# 5. Chunk the documents

In [18]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print("Chunks:", len(chunks))

Chunks: 73


# 6. Create embeddings

In [19]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# 7. Create Chroma Vector Store

In [20]:
vector_store = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    collection_name="customer_support"
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

# 8. Add Groq

In [21]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# 9. Create the RAG prompt

In [22]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful customer support assistant.

Answer ONLY using the context below.

If the answer is not in the context, say:
"I don't know based on the provided information."

Context:
{context}

Question:
{question}

Answer:
""")

# 10. Create the RAG function

In [23]:
def ask_rag(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    response = llm.invoke(
        prompt.format(
            context=context,
            question=question
        )
    )

    sources = set(
        f"{doc.metadata['source']} - Page {doc.metadata['page'] + 1}"
        for doc in docs
    )

    return response.content, "\n".join(sources)

In [24]:
answer, sources = ask_rag(
    "What is the return policy?"
)

print("ANSWER:\n", answer)
print("\nSOURCES:\n", sources)

ANSWER:
 Our return policy allows you to return an item within a certain timeframe, usually 30 days, but it varies by product and seller. You can initiate the return process by going to "Your Orders," selecting the item, and choosing "Return or Replace Items." Refunds are issued after the returned item is received and inspected. 

Additionally, you can return gifts using the gift receipt or by contacting Customer Support. However, some items like digital items, perishable goods, and certain hygiene products cannot be returned. It's also recommended to keep the original packaging for faster processing, but it's not always required.

SOURCES:
 FAQs.pdf - Page 5
FAQs.pdf - Page 4


# 11. Add Gradio

In [25]:
import gradio as gr

def chatbot(question):

    if not question.strip():
        return "Please enter a question."

    answer, sources = ask_rag(question)

    return f"""
### 🤖 Answer

{answer}

### 📚 Sources

{sources}
"""

In [26]:
with gr.Blocks() as demo:

    gr.Markdown("# ShopAssist AI")
    gr.Markdown("Customer Support RAG Assistant")

    question = gr.Textbox(
        label="Ask a question",
        placeholder="Can I return a product?"
    )

    button = gr.Button("Ask", variant="primary")

    output = gr.Markdown()

    button.click(
        chatbot,
        question,
        output
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9682da91a5239140dc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
